# Training application model

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path
from anomaly_detection.parser.data_parsers import parse_application
from anomaly_detection.transformers import application_transformer

In [ ]:
# Path to the data
project_folder = Path.cwd().parent

output_path = project_folder / "data/training/application"

model_path = project_folder / "src/anomaly_detection/models"

transformer_path = project_folder / "src/anomaly_detection/storage/transformers"

application_evtx_path = project_folder / "data/raw/93_applog.evtx"

application_evtx_path

# Data preparation

In [ ]:
application_training_records = parse_application(application_evtx_path)

application_training_records[0]

In [ ]:
flat_application_training_record = [{**record['timestamp'], **record['profile'], **record['data']} for record in application_training_records]

flat_application_training_record[0]

In [ ]:
application_training_df = pd.DataFrame(flat_application_training_record)

application_training_df.head(10)

In [ ]:
import dill

with open(transformer_path / "application_transformer.pkl", "wb") as f:
    dill.dump(application_transformer, f)

In [ ]:
X_application = application_transformer.transform(application_training_df)

X_application[0]

In [ ]:
X_application = np.nan_to_num(X_application)

X_application[0]

In [ ]:
np.savetxt(output_path / "application_features.csv", X_application, delimiter=",", fmt='%.18e')

In [ ]:
X_application = np.loadtxt(output_path / "application_features.csv", delimiter=",")

In [ ]:
pd.DataFrame(X_application).describe().T

# Modeling

In [ ]:
from sklearn.ensemble import IsolationForest

application_model = IsolationForest(
    n_estimators=512,
    contamination='auto',
    max_features=0.5,
    max_samples=512,
    n_jobs=-1,
)

application_model.fit(X_application)

In [ ]:
scores = application_model.score_samples(X_application)
predictions = application_model.predict(X_application)

In [ ]:
results_df = application_training_df.copy()

results_df["anomaly_score"] = scores
results_df["prediction"] = predictions

results_df.head()

In [ ]:
from anomaly_detection.utils.score_diagnostics import run_diagnostics

run_diagnostics(application_model, X_application, known_anomaly_mask=None)

In [ ]:
decision_scores = application_model.decision_function(X_application)

new_contamination = np.mean(decision_scores < 0.05)

new_contamination

In [ ]:
anomalies_df = results_df[results_df['prediction'] == -1]

anomalies_df

In [ ]:
anomalies_df['event_id'].value_counts()

In [ ]:
# Extract samples from each anomalous event ID
def pick_application_samples(n_samples_per_event):
    for event_id, group in anomalies_df.groupby("event_id"):
        print(f"\n{'=' * 60}")
        print(f"EventID {event_id}  --  {len(group)} anomalous rows")
        
        idx_min, idx_max = group.index.min(), group.index.max()
        idx_span = idx_max - idx_min
        print(f"row-index range: {idx_min} -> {idx_max}  (span: {idx_span})")
        print(f"{'=' * 60}")
    
        sample = group.sort_values("anomaly_score").head(n_samples_per_event)
    
        for _, row in sample.iterrows():
            print(f"\n  event_record_id: {row['event_record_id']}")
            print(f"  anomaly_score:    {row['anomaly_score']:.4f}")
 
            for col in [
                "event_id",
                "previous_event_id",
                "event_id_frequency",
                "version",
                "correlation_activity_id",
                "execution_thread_id",
            ]:
                if col in row and pd.notna(row[col]):
                    val = str(row[col])[:300]
                    print(f"  {col:<16}: {val}")
            
            if "context" in row:
                print(f"  context:          {str(row['context'])[:300]}")
            if "entity" in row:
                print(f"  entity:           {row['entity']}")
 

In [ ]:
from contextlib import redirect_stdout

with open(output_path / 'anomalous_samples.txt', 'w') as f:
    with redirect_stdout(f):
        pick_application_samples(5)

In [ ]:
import joblib

joblib.dump(application_model, model_path / "application_model.joblib")